# ДЗ 1. Классификация новостей по темам: сравнение методов векторизации

---


Классификация текстов — одна из частых задач, которую решает NLP-инженер: разложить входящий поток новостей, обращений пользователей,
тикетов или документов по категориям. Прежде чем тянуться к тяжелым моделям, в продуктовой разработке почти всегда сначала поднимают быстрый и интерпретируемый бейзлайн на классических методах — и часто его качества уже достаточно.

В этом ДЗ вы пройдете этот путь целиком на реальных новостях Ленты.ру: от работы с сырыми текстами до обоснованного выбора подхода. Главный результат — не одна лучшая модель, а аргументированное сравнение двух способов векторизации (частотного и на эмбеддингах) для последующей классификации
и вывод о том, какой из них и почему стоит брать под эту задачу.

### Что нужно сделать
1. Загрузить датасет новостей и посмотреть на распределение классов (блок 1).
2. Применить готовую предобработку и разбить данные на train / val / test (блок 2).
3. Обучить два подхода: `TF-IDF + классификатор` и `усредненные word-эмбеддинги (BOE) + классификатор` (блоки 3—4).
4. Оценить оба подхода на основании метрик, подходящих для дисбаланса классов, разобрать ошибки (блок 5).
5. Сформулировать обоснованный вывод о применимости каждого подхода (блок 6).

> **Сколько займет:** ориентировочно 4—6 часов.

> **Про объем кода:** не превращайте ноутбук в полотно, пишите функциями, комментируйте код. Лаконичное и читаемое решение оценивается выше длинного и запутанного.

### Оценивание

Максимальный балл за работу — 10. Баллы за каждый блок указаны в его заголовке.




## Настройка окружения

In [ ]:
#!pip install datasets gensim pymorphy3 nltk scikit-learn

In [1]:
import re
from functools import lru_cache

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
import pymorphy3

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

from gensim.models import Word2Vec

nltk.download("stopwords", quiet=True)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Окружение готово.")

Окружение готово.


## 1. Загрузка данных и EDA (1 балл)

**Датасет:** [`zloelias/lenta-ru`](https://huggingface.co/datasets/zloelias/lenta-ru) — новости Ленты.ру, 5 тематических рубрик. Всего в датасете ~207 тыс. строк, разложенных по двум сплитам: `train` (186 тыс.) и `test` (20.7 тыс.). Мы берем `train`.

**Колонки:** `title` (заголовок), `text` (текст новости), `topic` (тема, 5 классов), `labels` (число 0–4).

> **Почему мы не используем `title`.** Заголовок новости часто почти прямо называет рубрику, и с ним задача становится вырожденной: модель учится читать заголовок, а не текст. Мы сознательно оставляем только `text` — это ближе к реальной задаче классификации входящего потока.

**Пояснение.** Мы решаем многоклассовую задачу (5 тем). Обратите внимание на распределение
классов: темы встречаются неравномерно. Именно из-за этого позже мы будем смотреть не
на Accuracy, а на macro-F1 и на метрики по каждому классу — иначе модель,
угадывающая только частые темы, будет казаться хорошей.

**Про размер выборки.** Полный корпус считался бы слишком долго, поэтому ниже берется стратифицированная подвыборка в 20 тыс. строк. Это нормальная инженерная практика на этапе прототипа: сначала быстрый цикл экспериментов на подвыборке, потом финальный прогон на всем объеме.


In [ ]:
# Загрузка датасета и стратифицированная подвыборка
from datasets import load_dataset

SUBSAMPLE_SIZE = 20_000

ds = load_dataset("zloelias/lenta-ru", split="train")
df = ds.to_pandas()[["text", "topic", "labels"]].dropna().reset_index(drop=True)

df, _ = train_test_split(df, train_size=SUBSAMPLE_SIZE, random_state=RANDOM_STATE, stratify=df["labels"])
df = df.reset_index(drop=True)

# Соответствие меток и названий тем: понадобится в отчетах и при разборе ошибок, иначе classification_report напечатает только метки 0...4.
LABEL_NAMES = df.sort_values("labels")["topic"].unique().tolist()

print("Размер подвыборки:", df.shape)
print("\nРаспределение тем:")
print(df["topic"].value_counts())
print("\nLABEL_NAMES (индекс = метка):", LABEL_NAMES)

### ✍️ Задание 1.1. Графики

Постройте два графика:

1. Распределение классов (bar plot) — оцените дисбаланс глазами.
2. Распределение длины текста в словах (гистограмма) — пригодится при выборе `max_features` и при анализе ошибок.

Требования: у каждого графика должны быть подписаны оси и заголовок.

In [ ]:
# ✍️ ВАШ КОД (задание 1.1)
# Постройте два графика: распределение классов и распределение длины текста в словах.
# Подсказка: длину удобно посчитать так -> df["word_count"] = df["text"].str.split().str.len()

### 💬 Задание 1.2. Выводы

Ответьте своими словами и с числами с графиков:

1. Какой класс самый частый и какой самый редкий? Назовите их доли и отношение между ними.
2. Что видно по распределению длин текстов и как это повлияет на ваши дальнейшие решения?
3. Почему при таком распределении классов Accuracy — плохой единственный ориентир и что дает macro-F1?

> *Ваш ответ:*
> 1. ...
> 2. ...
> 3. ...

---
## 2. Предобработка текста и разбиение на выборки (1 балл)

Ниже дана готовая функция предобработки. Шаги стандартные: нижний регистр, токенизация, удаление стоп-слов и коротких токенов, лемматизация, повторный фильтр стоп-слов.

Разберитесь, что делает каждая строка, это может пригодиться, когда вы будете описывать, на каких признаках работает модель.

Два момента, на которые стоит обратить внимание:

- **`@lru_cache` на лемматизаторе.** Без него лемматизация 20 тыс. новостей заняла бы порядка 10 минут, с ним — около минуты.
- **Фильтр стоп-слов применяется дважды** — до и после лемматизации. После лемматизации словоформы меняются и могут попасть в стоп-слова (`были` не попадает в стоп-лист как есть, но лемма `быть` — попадает).


In [ ]:
# Предобработка
morph = pymorphy3.MorphAnalyzer()
stop_ru = set(stopwords.words("russian"))

# Токенизатор берет только кириллицу: цифры, латиница и пунктуация отбрасываются
TOKEN_RE = re.compile(r"[а-яё]+")

@lru_cache(maxsize=None)
def _lemma(token: str) -> str:
    """Лемматизация слова."""
    return morph.parse(token)[0].normal_form

def preprocess(text: str) -> str:
    """Нижний регистр -> токенизация -> стоп-слова и короткие токены -> лемматизация -> строка лемм."""
    tokens = TOKEN_RE.findall(text.lower())
    lemmas = [_lemma(t) for t in tokens if len(t) > 2 and t not in stop_ru]
    return " ".join(lem for lem in lemmas if lem not in stop_ru)

In [ ]:
# Применение предобработки
df["clean"] = df["text"].apply(preprocess)

print("Средняя длина до предобработки: ", round(df["text"].str.split().str.len().mean(), 1), "слов")
print("Средняя длина после предобработки: ", round(df["clean"].str.split().str.len().mean(), 1), "лемм")
print("\nПример:\n")
print("ДО: ", df.loc[0, "text"][:250])
print("\nПОСЛЕ: ", df.loc[0, "clean"][:250])

Разбиение на train / val / test со стратификацией по классу дано готовым.

> **Важное правило этого ДЗ.** Любой векторайзер (TF-IDF) и любые эмбеддинги обучаются **только на train**. Если обучить их на всех данных до разбиения, то будет утечка, и ваши метрики станут завышенными и недостоверными.
>
> В подходе А от этого защищает `Pipeline`: когда вы вызываете `pipe.fit(X_train, y_train)`, `fit` векторайзера видит только обучающую часть.
>
> А вот в подходе Б вы обучаете `Word2Vec` руками, и ничто не мешает случайно подать в него `X` целиком. Следите за этим сами: на вход должны идти только `tokens_train`.

**Роли выборок:** `train` — обучение, `val` — подбор конфигурации, `test` — однократная финальная оценка. Не подбирайте ничего по test: как только вы выбрали конфигурацию по тестовой метрике, test перестал быть честной оценкой.

In [ ]:
# Разбиение 70 / 15 / 15 со стратификацией
X = df["clean"]
y = df["labels"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp)

print("train / val / test:", len(X_train), "/", len(X_val), "/", len(X_test))

### 💬 Задание 2.1. Вопрос на понимание

Представьте, что вы обучили `TfidfVectorizer` на **всех** текстах (`X`) еще до разбиения, а потом уже поделили матрицу признаков на train и test.

Ответьте в 2–4 предложениях, что именно здесь утекает из test в train и почему итоговая метрика окажется завышенной.

> *Ваш ответ:* ...

## 3. Подход А: TF-IDF + классификатор (2 балла)

Сначала две готовые вспомогательные функции. `evaluate` печатает per-class отчет, рисует матрицу ошибок и складывает метрики в словарь `RESULTS` — за счет этого сводная таблица в блоке 5 соберется сама.


In [ ]:
# Вспомогательные функции: отчет по метрикам и топ-признаки
RESULTS = {}

def evaluate(name, y_true, y_pred, plot_cm=True):
    """Печатает per-class отчет, рисует матрицу ошибок, сохраняет метрики в RESULTS."""
    print(f"=== {name} ===")
    print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=3))

    RESULTS[name] = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro-F1": f1_score(y_true, y_pred, average="macro"),
        "weighted-F1": f1_score(y_true, y_pred, average="weighted"),
    }

    if plot_cm:
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(6.5, 5))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
        plt.xlabel("Предсказано")
        plt.ylabel("Истина")
        plt.title(name)
        plt.tight_layout()
        plt.show()

    return RESULTS[name]


def top_features(pipe, vec_step="tfidf", clf_step="clf", n=15):
    """Топ-n слов с наибольшим весом для каждого класса (для линейных моделей)."""
    names = np.array(pipe.named_steps[vec_step].get_feature_names_out())
    coefs = pipe.named_steps[clf_step].coef_
    for i, cls in enumerate(LABEL_NAMES):
        top = names[np.argsort(coefs[i])[-n:][::-1]]
        print(f"{cls:>18} | {", ".join(top)}")

### ✍️ Задание 3.1. Pipeline и подбор конфигурации

Соберите `Pipeline`: `TfidfVectorizer` c `LogisticRegression`, переберите **все** конфигурации из готового списка `CONFIGS` и выберите лучшую **по macro-F1 на val**.

- Список конфигураций зафиксирован намеренно: полноценный grid search здесь не нужен и только съест ваше время. Весь перебор занимает пару минут.
- `LogisticRegression` берите с `max_iter=1000`. С дефолтным `max_iter=100` на TF-IDF вы почти наверняка получите `ConvergenceWarning` — это сигнал, что модель не успела сойтись.
- Для каждой конфигурации напечатайте ее macro-F1 на val, а в конце явно определите победителя.

In [ ]:
# ✍️ ВАШ КОД (задание 3.1)
CONFIGS = [
    dict(max_features=20_000, ngram_range=(1, 1), min_df=2),
    dict(max_features=50_000, ngram_range=(1, 1), min_df=2),
    dict(max_features=50_000, ngram_range=(1, 1), min_df=5),
    dict(max_features=50_000, ngram_range=(1, 2), min_df=5),
]

# Переберите CONFIGS, для каждой обучите Pipeline на train и посчитайте macro-F1 на val.
# Сохраните лучший обученный пайплайн в переменную pipe_tfidf — он понадобится дальше.

### ✍️ Задание 3.2. Финальная оценка на test

Оцените **лучший** пайплайн на test с помощью готовой функции `evaluate`. Это единственный раз, когда вы трогаете test в этом блоке.

In [ ]:
# ✍️ ВАШ КОД (задание 3.2)

### ✍️ Задание 3.3. Топ-признаки

Выведите топ-слова по классам с помощью готовой функции `top_features`. Частотные методы интерпретируемы — воспользуйтесь этим.

In [ ]:
# ✍️ ВАШ КОД (задание 3.3)

### 💬 Задание 3.4. Комментарий к топ-признакам
Возьмите **минимум две темы** и разберите их топ-слова. Обязательно называйте конкретные слова из вашей выдачи.

1. Какие слова осмысленны, то есть действительно характеризуют тему?
2. Какие слова оказались артефактами корпуса — имена собственные, названия агентств, служебная лексика редакции? Почему они стали важны для модели?
3. Что это говорит о том, на что модель реально опирается при решении?

> *Ваш ответ:* ...

## 4. Подход Б: усредненные word-эмбеддинги (BOE) + классификатор (2 балла)

**Пояснение.** Усреднение векторов слов (Bag of Embeddings, BOE из лекции 3) — самый простой способ получить вектор целого документа, и он намеренно наивный. Запомните, где он проседает: порядок слов и контекст полностью теряются: `"Динамо" обыграл "Спартак"` и `"Спартак" обыграл "Динамо"` дают один и тот же вектор.

> **Важно.** `Word2Vec` должен видеть **только train**.


### ✍️ Задание 4.1. Обучите Word2Vec на train

Обучите `Word2Vec` на токенах **обучающей** выборки. Гиперпараметры возьмите такие: `vector_size=200`, `window=5`, `min_count=3`, `sg=1`, `epochs=5`, `seed=RANDOM_STATE`, `workers=1`.

- `workers=1` нужен для воспроизводимости: при нескольких потоках порядок обновлений весов недетерминирован, и один только `seed` одинакового результата **не** гарантирует.
- После обучения напечатайте размер словаря и проверьте модель на вменяемость через `most_similar` для любого частотного слова.

In [ ]:
# ✍️ ВАШ КОД (задание 4.1)

### ✍️ Задание 4.2. Реализуйте вектор документа

Реализуйте функцию `doc_vector`: вектор документа = **среднее** векторов его слов. Слова, которых нет в словаре модели, пропускаем.

Обязательно обработайте краевой случай: если **ни одного** слова документа нет в словаре, функция должна вернуть нулевой вектор нужной размерности, а не упасть и не вернуть `nan`.

> Подсказка: у обученной модели `Word2Vec` векторы лежат в `model.wv`, проверка наличия слова — `t in model.wv`, сам вектор - `model.wv[t]`.

In [ ]:
# ✍️ ВАШ КОД (задание 4.2)
def doc_vector(tokens, model, dim):
    """Средний вектор по словам документа; нули, если ни одного слова нет в словаре."""
    raise NotImplementedError

### ✍️ Задание 4.3. Векторизуйте, обучите, оцените

1. Векторизуйте train / val / test через `doc_vector`.
2. **Проверьте, что векторизация сработала:** напечатайте форму матрицы и долю документов с нулевым вектором. Если нулевых векторов много — что-то пошло не так, и дальше считать бессмысленно.
3. Обучите `LogisticRegression(max_iter=1000)` на train и оцените на test через `evaluate`.

> Этот пункт с проверкой на нули — не формальность. Самая частая и самая дорогая ошибка в этом задании выглядит так: словарь эмбеддингов не совпал с вашими токенами, все векторы стали нулевыми, и модель предсказывает один класс. Проверяйте промежуточный результат.

In [ ]:
# ✍️ ВАШ КОД (задание 4.3)

## 5. Сравнение и анализ ошибок (2 балла)


### ✍️ Задание 5.1. Сводная таблица

Сведите результаты обоих подходов в одну таблицу: Accuracy, macro-F1, weighted-F1.

> Если вы оба раза вызывали `evaluate`, метрики уже лежат в словаре `RESULTS` — таблица собирается одной строкой.

In [ ]:
# ✍️ ВАШ КОД (задание 5.1)

### ✍️ Задание 5.2. 3 примера ошибок

Найдите **3 примера**, на которых лучшая модель ошибается. Для каждого выведите: фрагмент текста, истинную тему и предсказанную.

In [ ]:
# ✍️ ВАШ КОД (задание 5.2)

### 💬 Задание 5.3. Разбор

1. По каждому из 3 примеров: ваша гипотеза, почему модель ошиблась именно здесь. Опирайтесь на содержание конкретного текста — какие слова могли «увести» модель. Ответы уровня «текст сложный» не засчитываются.
2. По матрице ошибок: какие две темы путаются между собой чаще всего? Назовите число из матрицы и предположите причину.

> *Ваш ответ:*
> 1. ...
> 2. ...

## 6. Вывод (2 балла)






### 💬 Задание 6.1. Вывод

Сформулируйте 5—8 предложений своими словами, опираясь на числа из вашей таблицы:

- Какой подход к векторизации показал себя лучше на этой задаче и почему?
- В чем сильные и слабые стороны каждого — качество, скорость, интерпретируемость, поведение на дисбалансе?
- Какой подход вы бы взяли в реальный сервис рубрикации и при каких условиях изменили бы решение?
- Что можно сделать дальше для роста качества?

> *Ваш ответ:* ...

### 💬 Задание 6.2. Честность сравнения

Посмотрите на то, как был устроен ваш эксперимент. Подход А вы подбирали по val, перебрав 4 конфигурации. Подход Б взяли с фиксированными гиперпараметрами, без подбора, а `Word2Vec` обучили на корпусе всего в ~14 тыс. коротких документов.

В 2–4 предложениях расскажите, какая часть разрыва между подходами объясняется **природой методов**, а какая — **условиями эксперимента**. Что конкретно вы бы изменили, чтобы сравнение стало честным?

> *Ваш ответ:* ...

## Бонусное задание

> Бонусная часть не является обязательной к выполнению и не влияет на оценку за домашнее задание. Выполнив это задание, вы получите развернутую обратную связь в свободной форме. Здесь нет строгих критериев выполнения.

В основной части домашнего задания вы сравнивали два способа векторизации при одном и том же классификаторе. Здесь вы сделаете обратное: зафиксируете векторизацию (TF-IDF) и поменяете классификатор — возьмете наивный байесовский классификатор.






### ✍️ Задание Б1. Обучите MultinomialNB

Соберите точно такой же `Pipeline`, как в Задании 3.1, но с `MultinomialNB()` вместо `LogisticRegression`. Параметры `TfidfVectorizer` возьмите те, что оказались лучшими в ваших экспериментах.

Обучите на train, оцените на test через готовую функцию `evaluate` и выведите сводную таблицу по обоим классификаторам (напомним, что `RESULTS` уже накапливает метрики — таблица собирается одной строкой).

In [ ]:
# ✍️ ВАШ КОД (задание Б1)
# 1. Pipeline: TfidfVectorizer(**best_params) -> MultinomialNB()
# 2. Обучите на train, оцените на test через evaluate(...)
# 3. Выведите сводную таблицу по обоим классификаторам
# Сохраните обученный пайплайн в переменную pipe_nb — он понадобится дальше

### 💬 Задание Б2. Сравните результаты

В лекции о классификации мы говорили о том, что на практике часто достаточно дискриминативного подхода, и при достаточном количестве данных он нередко работает лучше. Логистическая регрессия — дискриминативная модель, наивный Байес — генеративная.

1. Подтвердился ли этот тезис на ваших данных? Сравните по macro-F1, а не только по Accuracy.
2. Посмотрите на per-class отчет. На каких классах разрыв между моделями самый большой, а на каких его почти нет?
3. Выполните ячейку ниже и посмотрите на априорные вероятности классов, которые выучил наивный Байес. Как они связаны с тем, что вы увидели в пункте 2?

> *Ваш ответ:*
>
> 1. ...
> 2. ...
> 3. ...

In [ ]:
# Априорные вероятности классов, выученные наивным Байесом
priors = np.exp(pipe_nb.named_steps["clf"].class_log_prior_)
for cls, p in zip(LABEL_NAMES, priors):
    print(f"{cls:>18} | P(класс) = {p:.3f}")

### ✍️ Задание Б3. Посмотрите на топ-признаки для наивного Байеса

В основной части ДЗ вы смотрели топ-признаки логистической регрессии через `coef_`. Попробуем сделать то же самое для наивного Байеса.

Для этого необходимо адаптировать нашу функцию `top_features`: у `MultinomialNB` нет атрибута `coef_`, потому что модель устроена иначе: она не подбирает веса признаков, а оценивает вероятности. Найдите в документации библиотеки `scikit-learn` необходимый атрибут, который позволит посмотреть на топ-признаки, и напишите свою версию функции `top_features_nb`, которая выведет топ-15 слов для каждого класса у наивного Байеса.

In [ ]:
# ✍️ ВАШ КОД (задание Б3)

# Напишите свою версию для наивного Байеса:
def top_features_nb(pipe, n=15):
    """Топ-n слов по log P(слово | класс) для каждого класса."""
    raise NotImplementedError

### 💬 Задание Б4. Оцените топ-признаки для наивного Байеса

Посмотрите внимательно на то, что напечатала ваша функция `top_features_nb`, и сравните с топом логистической регрессии выше в основной части ДЗ.

1. Сравните списки разных классов между собой — что вы замечаете?
2. Топ-признаки у наивного Байеса сортируются по величине `log P(слово | класс)`. Почему именно такая сортировка дает такой результат? Подумайте, какие слова обязательно имеют высокую вероятность внутри любого класса.
3. В чем принципиальная разница между тем, что показывает `coef_` логистической регрессии, и тем, что показывает найденный вами атрибут для выделения топ-признаков у наивного Байеса? Свяжите ответ с различием генеративных и дискриминативных моделей из лекции о классификации.

> *Ваш ответ:*
>
> 1. ...
> 2. ...
> 3. ...